# Preview notebook only

This notebook is for inspecting already-generated nonmodel-attribute outputs and trying temporary table previews. The production source is `main.py`, run through `code/Makefile`. Do not use this notebook to regenerate paper artifacts.


# Nonmodel Attributes: Table 9 Playground

This notebook is for trying Table 9 specifications without editing the production script. The editable cell below controls the variables, interaction rows, samples, and whether citation-weighted versions are run. By default it only previews results in the notebook and does not overwrite files in `../output/`.


In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
from patsy import dmatrices
from statsmodels.robust import norms, scale

TASK_DIR = Path.cwd().parent if Path.cwd().name == "code" else Path("tasks/measurement/nonmodel_attributes").resolve()
INPUT_DIR = TASK_DIR / "input"
OUTPUT_DIR = TASK_DIR / "output"
PLAY_OUTPUT_DIR = OUTPUT_DIR / "notebook_preview"

sys.path.append(str(TASK_DIR.parent))
from regression_table_tools import (
    apply_clustered_inference,
    build_table,
    cluster_groups,
    fit_model,
    formula_for,
    independent_design,
    load_regression_data,
    load_table_params,
    pseudo_r2,
    stars,
)

SPEC_KEY = "nonmodel_attributes"
params = load_table_params(TASK_DIR / "code" / "main.py")
base_spec = params["table_specs"][SPEC_KEY]
df = load_regression_data(INPUT_DIR, params)

samples = {
    "full_sample": df,
    "estimated_models": df.loc[df["estimated"] == 1].copy(),
}

print(f"Rows after timing restrictions: {len(df)}")
print(f"Rows in estimated-model sample: {len(samples['estimated_models'])}")
print(f"Cluster variable: {params.get('cluster_var')}")


## Editable Specification

Change this cell to try different Table 9 designs. `model_terms` is the right-hand-side list added on top of the standard policy-rule and estimated-model controls in `config/params.yaml`. Use Patsy interaction syntax, for example `"hh_demand:bank"`.


In [ ]:
# Main attributes currently used for Table 9.
attribute_terms = [
    "cb_authors_ext",
    "ln_neq",
    "vint_mid",
    "vint_late",
]

# Interaction candidates currently used for Table 9. Comment rows in/out freely.
interaction_terms = [
    "cb_authors_ext:vint_late",
]

# This is the actual specification sent to Patsy/statsmodels.
model_terms = attribute_terms + interaction_terms

# Labels and row order for the manuscript-style preview table.
display_terms = [
    ("cb_authors_ext", "Central bank authors"),
    ("ln_neq", "Ln(Number of equations)"),
    ("vint_mid", "Middle vintage"),
    ("vint_late", "Late vintage"),
    ("cb_authors_ext:vint_late", "Central bank authors*Late vintage"),
]

# Run choices. Keep this small while experimenting, then expand as needed.
samples_to_run = ["full_sample", "estimated_models"]
citation_weighted_versions = [False, True]

# Keep False while experimenting. If True, preview .tex/.csv files are written to output/notebook_preview/.
write_preview_outputs = False

print("Model terms:")
for term in model_terms:
    print("  ", term)


## Fit Helpers

These cells mirror the production Table 9 logic, but use `model_terms` from the editable cell instead of reading the spec from `config/params.yaml`. Standard errors are clustered by `model` in both unweighted and citation-weighted versions.


In [ ]:
outcome_order = [
    ("IScurve20", "\\textit{y-slope}"),
    ("infl_per_rr20", "\\textit{$\\pi$-slope}"),
    ("sacratio20", "\\textit{sacratio}"),
    ("y_timing_max", "\\textit{y-timing}"),
    ("piq_timing_max", "\\textit{$\\pi$-timing}"),
]

preview_params = dict(params)
preview_spec = dict(base_spec)
preview_spec["variables"] = list(model_terms)
preview_params["table_specs"] = dict(params["table_specs"])
preview_params["table_specs"][SPEC_KEY] = preview_spec


def adjusted_r2(r2_value, nobs, nparams):
    # Standard finite-sample adjustment for the fit statistic reported in each column.
    if not np.isfinite(r2_value) or nobs <= nparams:
        return np.nan
    return 1.0 - (1.0 - r2_value) * (nobs - 1.0) / (nobs - nparams)


def weighted_design(depvar, sample_df):
    formula = formula_for(depvar, model_terms, sample_df, preview_params)
    y, x = dmatrices(formula, data=sample_df, return_type="dataframe")
    weights = pd.to_numeric(sample_df.loc[x.index, "citation_weight"], errors="coerce")
    keep = weights.notna() & np.isfinite(weights) & (weights > 0)
    y = y.loc[keep].copy()
    x = x.loc[keep].copy()
    weights = weights.loc[keep].astype(float)
    weights = weights / weights.mean()
    y, x = independent_design(y, x)
    return y, x, weights.loc[x.index]


def fit_citation_weighted_model(depvar, sample_df):
    y, x, weights = weighted_design(depvar, sample_df)
    groups = cluster_groups(sample_df, x, preview_params)
    glm_fit_kwargs = {"cov_type": "HC0"}
    if groups is not None:
        glm_fit_kwargs = {"cov_type": "cluster", "cov_kwds": {"groups": groups}}

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        # Timing outcomes are count models; keep the same overdispersion fallback as production.
        if depvar in preview_params["timing_outcomes"]:
            poisson = sm.GLM(y, x, family=sm.families.Poisson(), freq_weights=weights).fit(**glm_fit_kwargs)
            mu = poisson.fittedvalues.squeeze()
            aux_y = ((y.squeeze() - mu) ** 2) - y.squeeze()
            aux_x = mu ** 2
            aux = sm.WLS(aux_y, aux_x, weights=weights).fit(cov_type="HC0")
            alpha_hat = float(aux.params.squeeze()) if np.isfinite(aux.params.squeeze()) else 0.0
            p_over = float(aux.pvalues.squeeze()) if np.isfinite(aux.pvalues.squeeze()) else 1.0
            if p_over < 0.05 and alpha_hat > 0:
                nb = sm.families.NegativeBinomial(alpha=alpha_hat)
                fit = sm.GLM(y, x, family=nb, freq_weights=weights).fit(**glm_fit_kwargs)
            else:
                fit = poisson
            fit.fittedvalues_original = fit.fittedvalues.copy()
            fit.citation_weights = weights.copy()
            fit.nobs_unweighted = int(y.shape[0])
            return fit

        # RLM has no citation-weight argument, so use the usual sqrt(weight) transform.
        sqrt_weights = np.sqrt(weights)
        weighted_y = y.squeeze().mul(sqrt_weights, axis=0)
        weighted_x = x.mul(sqrt_weights, axis=0)
        fit = sm.RLM(weighted_y, weighted_x, M=norms.TukeyBiweight(c=4.685)).fit(
            scale_est=scale.HuberScale(),
            update_scale=True,
            cov="H1",
            conv="coefs",
        )
        fit = apply_clustered_inference(fit, weighted_x, weighted_y, groups, weights=fit.weights)
        fit.fittedvalues_original = pd.Series(
            np.asarray(x, dtype=float) @ fit.params.to_numpy(dtype=float),
            index=x.index,
        )
        fit.citation_weights = weights.copy()
        fit.nobs_unweighted = int(y.shape[0])
        return fit


In [ ]:
def ordinary_r2_stats(depvar, sample_df, citation_weighted=False):
    # Elasticity columns report ordinary R-squared diagnostics for the same design matrix.
    if citation_weighted:
        y, x, weights = weighted_design(depvar, sample_df)
        model = sm.WLS(y.squeeze(), x, weights=weights).fit()
    else:
        formula = formula_for(depvar, model_terms, sample_df, preview_params)
        y, x = dmatrices(formula, data=sample_df, return_type="dataframe")
        y, x = independent_design(y, x)
        model = sm.OLS(y.squeeze(), x).fit()
    return float(model.rsquared), float(model.rsquared_adj)


def robust_weighted_r2(fit, sample_df, depvar):
    # Combine citation weights, when present, with RLM robustness weights.
    if not hasattr(fit, "weights"):
        return np.nan
    if hasattr(fit, "fittedvalues_original"):
        y = pd.to_numeric(sample_df.loc[fit.fittedvalues_original.index, depvar], errors="coerce").to_numpy(dtype=float)
        fitted = np.asarray(fit.fittedvalues_original, dtype=float)
        weights = np.asarray(fit.weights, dtype=float) * np.asarray(fit.citation_weights, dtype=float)
    else:
        y = pd.to_numeric(sample_df.loc[fit.fittedvalues.index, depvar], errors="coerce").to_numpy(dtype=float)
        fitted = np.asarray(fit.fittedvalues, dtype=float)
        weights = np.asarray(fit.weights, dtype=float)
    ybar = np.average(y, weights=weights)
    denom = np.sum(weights * (y - ybar) ** 2)
    if denom <= 0:
        return np.nan
    return 1.0 - np.sum(weights * (y - fitted) ** 2) / denom


def fit_see(fit, sample_df, depvar):
    if hasattr(fit, "fittedvalues_original"):
        y = pd.to_numeric(sample_df.loc[fit.fittedvalues_original.index, depvar], errors="coerce").to_numpy(dtype=float)
        resid = y - np.asarray(fit.fittedvalues_original, dtype=float)
        weights = np.asarray(fit.citation_weights, dtype=float)
    else:
        y = pd.to_numeric(sample_df.loc[fit.fittedvalues.index, depvar], errors="coerce").to_numpy(dtype=float)
        resid = y - np.asarray(fit.fittedvalues, dtype=float)
        weights = np.ones(len(y))
    df_resid = len(y) - len(fit.params)
    if df_resid <= 0:
        return np.nan
    return np.sqrt(np.sum(weights * resid ** 2) / df_resid)


def fit_r2(fit, sample_df, depvar):
    if hasattr(fit, "fittedvalues_original"):
        y = pd.to_numeric(sample_df.loc[fit.fittedvalues_original.index, depvar], errors="coerce").to_numpy(dtype=float)
        fitted = np.asarray(fit.fittedvalues_original, dtype=float)
        weights = np.asarray(fit.citation_weights, dtype=float)
        ybar = np.average(y, weights=weights)
        denom = np.sum(weights * (y - ybar) ** 2)
        if denom <= 0:
            return np.nan
        return 1.0 - np.sum(weights * (y - fitted) ** 2) / denom
    return pseudo_r2(fit, sample_df, depvar)


In [ ]:
def latex_coef(fit, term):
    if term not in fit.params.index:
        return ""
    coef = float(fit.params[term])
    p_value = float(fit.pvalues[term]) if term in fit.pvalues.index else np.nan
    coef_text = f"{coef:.2f}"
    if coef_text == "-0.00":
        coef_text = "0.00"
    coef_stars = stars(p_value)
    if coef_stars:
        return f"${coef_text}^{{{coef_stars}}}$"
    if coef < 0:
        return f"${coef_text}$"
    return coef_text


def latex_se(fit, term):
    if term not in fit.bse.index:
        return ""
    se_text = f"{float(fit.bse[term]):.2f}"
    if se_text == "-0.00":
        se_text = "0.00"
    return f"({se_text})"


def stat_number(value):
    if isinstance(value, str):
        return value
    if not np.isfinite(value):
        return "--"
    text = f"{value:.2f}"
    if text == "-0.00":
        text = "0.00"
    return text


def plain_coef(fit, term):
    if term not in fit.params.index:
        return ""
    return f"{float(fit.params[term]):.3f}{stars(float(fit.pvalues[term]))}"


def plain_se(fit, term):
    if term not in fit.bse.index:
        return ""
    return f"({float(fit.bse[term]):.3f})"


## Run Preview Tables

Rerun from the editable spec cell through this cell after changing variables. The displayed DataFrames are easier to scan in the notebook; the LaTeX blocks below show exactly what would go into the paper.


In [ ]:
def fit_preview_table(sample_df, citation_weighted=False):
    fits = {}
    diagnostics = {}
    rows = []

    for depvar, label in outcome_order:
        if citation_weighted:
            fit = fit_citation_weighted_model(depvar, sample_df)
        else:
            fit = fit_model(depvar, model_terms, sample_df, preview_params)
        fits[depvar] = fit
        nobs = float(fit.nobs_unweighted) if hasattr(fit, "nobs_unweighted") else float(fit.nobs)
        nparams = len(fit.params)

        if depvar in preview_params["timing_outcomes"]:
            r2_value = fit_r2(fit, sample_df, depvar)
            adj_value = adjusted_r2(r2_value, nobs, nparams)
            rw2_value = "--"
        else:
            r2_value, adj_value = ordinary_r2_stats(depvar, sample_df, citation_weighted=citation_weighted)
            rw2_value = robust_weighted_r2(fit, sample_df, depvar)

        diagnostics[depvar] = {
            "r2": r2_value,
            "adj_r2": adj_value,
            "rw2": rw2_value,
            "see": fit_see(fit, sample_df, depvar),
            "n": int(nobs),
        }

    displayed_terms = [
        (term, label_text)
        for term, label_text in display_terms
        if any(term in fits[depvar].params.index for depvar, _ in outcome_order)
    ]

    for term, label_text in displayed_terms:
        coef_row = {"row": label_text}
        se_row = {"row": ""}
        for depvar, label in outcome_order:
            coef_row[label] = plain_coef(fits[depvar], term)
            se_row[label] = plain_se(fits[depvar], term)
        rows.extend([coef_row, se_row])

    for stat_label, stat_key in [("R2", "r2"), ("Adj. R2", "adj_r2"), ("Rw2", "rw2"), ("S.E.E.", "see"), ("N", "n")]:
        stat_row = {"row": stat_label}
        for depvar, label in outcome_order:
            value = diagnostics[depvar][stat_key]
            stat_row[label] = str(int(value)) if stat_key == "n" else stat_number(value)
        rows.append(stat_row)

    return fits, diagnostics, displayed_terms, pd.DataFrame(rows)


preview_results = {}
for sample_name in samples_to_run:
    for citation_weighted in citation_weighted_versions:
        sample_df = samples[sample_name]
        key = (sample_name, citation_weighted)
        fits, diagnostics, displayed_terms_used, table_df = fit_preview_table(sample_df, citation_weighted=citation_weighted)
        preview_results[key] = {
            "fits": fits,
            "diagnostics": diagnostics,
            "displayed_terms": displayed_terms_used,
            "table_df": table_df,
        }
        weight_label = ", citation weighted" if citation_weighted else ""
        print(f"\n{sample_name.replace('_', ' ')}{weight_label}")
        display(table_df)


## Print LaTeX

This prints manuscript-style LaTeX for each preview. It does not write to production outputs unless `write_preview_outputs = True` in the editable cell.


In [ ]:
def build_latex(sample_name, citation_weighted, result):
    row_end = r" \\" 
    caption = "Effects of Non-Model Attributes on Macro Outcomes (full sample)"
    label = "tab:nonmodel_attributes"
    note_tail = "See Table~\\ref{tab:nonmodel_attributes_B} for statistics with estimated models only."
    output_stem = "table9_nonmodel_attributes_full_sample"
    if sample_name == "estimated_models":
        caption = "Effects of Non-Model Attributes on Macro Outcomes (estimated models only)"
        label = "tab:nonmodel_attributes_B"
        note_tail = "See Table~\\ref{tab:nonmodel_attributes} for full-sample statistics."
        output_stem = "table9_nonmodel_attributes_estimated_models"
    if citation_weighted:
        caption = caption.replace(")", ", citation weighted)")
        label = label + "_cw"
        output_stem = output_stem + "_citation_weighted"
        note_tail = note_tail + " Citation weights are age-adjusted, log-transformed, and normalized to mean one within each regression sample."

    if sample_name == "full_sample":
        controls_note = "Regressions include dummy variables for policy rules and for estimated models as controls. "
    else:
        controls_note = "Regressions include dummy variables for policy rules as controls. "

    fits = result["fits"]
    diagnostics = result["diagnostics"]
    displayed_terms_used = result["displayed_terms"]

    latex_lines = [
        "\\begin{table}[H]",
        "\\centering",
        f"\\caption{{ \\\\ \\underline{{{caption}}} }}",
        f"\\label{{{label}}}",
        "",
        "\\begin{tabular}{l c c c c c}",
        "\\toprule",
        "& \\multicolumn{3}{c}{\\textbf{Elasticities}$^{\\dagger}$}",
        "& \\multicolumn{2}{c}{\\textbf{Timing}$^{\\dagger\\dagger}$}" + row_end,
        "\\cmidrule(lr){2-4}\\cmidrule(lr){5-6}",
        "& \\textit{y-slope} & \\textit{$\\pi$-slope} & \\textit{sacratio}",
        "& \\textit{y-timing} & \\textit{$\\pi$-timing}" + row_end,
        "\\midrule",
        "",
    ]

    for row_index, (term, label_text) in enumerate(displayed_terms_used):
        latex_lines.append(label_text)
        latex_lines.append("& " + " & ".join(latex_coef(fits[depvar], term) for depvar, _ in outcome_order) + row_end)
        latex_lines.append("& " + " & ".join(latex_se(fits[depvar], term) for depvar, _ in outcome_order) + row_end)
        if row_index < len(displayed_terms_used) - 1:
            latex_lines.append("\\addlinespace[0.5em]")
            latex_lines.append("")

    latex_lines.extend(["", "\\midrule", ""])
    for row_label, stat_key in [("$R^{2}$", "r2"), ("$\\bar{R}^{2}$", "adj_r2"), ("$R_{w}^{2}$", "rw2"), ("S.E.E.", "see"), ("$N$", "n")]:
        cells = []
        for depvar, _ in outcome_order:
            value = diagnostics[depvar][stat_key]
            cells.append(str(int(value)) if stat_key == "n" else stat_number(value))
        latex_lines.append(row_label + " & " + " & ".join(cells) + row_end)

    latex_lines.extend([
        "",
        "\\bottomrule",
        "\\multicolumn{6}{p{14cm}}{\\footnotesize Notes: $\\dagger$ Robust least squares with standard errors clustered by model. See Table~\\ref{tab:baseline_reg} for details. $\\dagger\\dagger$ Timing columns are estimated by Poisson GLM, with a negative-binomial GLM used when an auxiliary overdispersion test rejects equidispersion. GLM covariances are clustered by model. See Table~\\ref{tab:baseline_reg} for details. " + controls_note + "Early vintage is the omitted vintage category. *, **, *** indicate statistical significance at 10, 5, and 1 percent, respectively. See Table~\\ref{tab:outcome_vars} for definitions of variables. " + note_tail + "}",
        "\\end{tabular}",
        "\\end{table}",
    ])
    return output_stem, "\n".join(latex_lines) + "\n"


latex_previews = {}
for key, result in preview_results.items():
    sample_name, citation_weighted = key
    output_stem, latex_text = build_latex(sample_name, citation_weighted, result)
    latex_previews[key] = latex_text
    print("\n" + "=" * 80)
    print(output_stem)
    print("=" * 80)
    print(latex_text)

    if write_preview_outputs:
        PLAY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        (PLAY_OUTPUT_DIR / f"{output_stem}.tex").write_text(latex_text, encoding="utf-8")
        result["table_df"].to_csv(PLAY_OUTPUT_DIR / f"{output_stem}.csv", index=False)
        print(f"Wrote preview files to {PLAY_OUTPUT_DIR}")


## Compare Against Production Outputs

Use this only when the editable spec matches the production Table 9 spec. It gives a quick sanity check against files generated by `make`.


In [ ]:
production_files = [
    "table9_nonmodel_attributes_full_sample.tex",
    "table9_nonmodel_attributes_estimated_models.tex",
    "table9_nonmodel_attributes_full_sample_citation_weighted.tex",
    "table9_nonmodel_attributes_estimated_models_citation_weighted.tex",
]

for filename in production_files:
    path = OUTPUT_DIR / filename
    print("\n" + filename)
    print(path.read_text(encoding="utf-8")[:1200])
